In [1]:
!pip install -q roboflow

from roboflow import Roboflow
import zipfile
import os
import shutil

# Download dataset
rf = Roboflow(api_key="1LXmrHIjxoP1M8QXA59l")
project = rf.workspace("urpc2020").project("urpc2020")
version = project.version(2)
dataset = version.download("yolov12")   # downloads to ./URPC2020-2

# Path to downloaded dataset folder
dataset_dir = "URPC2020-2"

# Create a ZIP file
zip_name = "urpc2020_dataset.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=os.path.dirname(dataset_dir))
            zipf.write(file_path, arcname)

print(f"✅ Dataset saved as {zip_name} (size: {os.path.getsize(zip_name) / (1024*1024):.2f} MB)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 42.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to URPC2020-2 in yolov12:: 100%|██████████| 37894/37894 [00:05<00:00, 7406.75it/s] 


✅ Dataset saved as urpc2020_dataset.zip (size: 1085.14 MB)


In [2]:
# %% [markdown]
# # YOLO11 + Novel Spatial‑wise Dynamic Fusion (SDF) – URPC2020
# ## Novelty: Adaptive spatial weighting of multi‑scale features
# Choose model size: n, s, m, l

# ------------------------------
# 1. Install
# ------------------------------
!pip install -q ultralytics matplotlib seaborn

# ------------------------------
# 2. Imports & Dataset Setup
# ------------------------------
import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from ultralytics import YOLO
from ultralytics.nn.modules import C2f

# ------------------------------
# 2b. Choose model size here
# ------------------------------
model_size = 'm'          # Options: 'n', 's', 'm', 'l'
model_name = f"yolo11{model_size}.pt"
print(f"Using model: {model_name}")

# Adjusted batch sizes to avoid OOM
batch_sizes = {'n': 16, 's': 10, 'm': 6, 'l': 4}
batch = batch_sizes.get(model_size, 10)

# Dataset Path (autodetect)
dataset_dir = "/kaggle/working/URPC2020-2"
data_yaml = os.path.join(dataset_dir, "data.yaml")

if not os.path.exists(data_yaml):
    for path in ["/kaggle/input/urpc2020/data.yaml", "/kaggle/input/urpc2020-2/data.yaml"]:
        if os.path.exists(path):
            data_yaml = path
            dataset_dir = os.path.dirname(path)
            break

with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)

print(f"✅ Classes: {data_cfg['names']} ({data_cfg['nc']} classes)")
print(f"Dataset: {dataset_dir}")

# ------------------------------
# 3. NOVEL FUSION BLOCK: Spatial‑wise Dynamic Fusion (SDF)
# ------------------------------
class SpatialDynamicFusion(nn.Module):
    """
    Novel: Learns a spatial attention map for each scale (1x1, 3x3, 5x5)
    and fuses them by per‑pixel weighted sum.
    """
    def __init__(self, channels, reduction=8):
        super().__init__()
        # Multi‑scale feature extractors (lightweight)
        self.conv1 = nn.Conv2d(channels, channels, 1, 1, 0)
        self.conv3 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.conv5 = nn.Conv2d(channels, channels, 5, 1, 2)
        
        # BatchNorm + activation
        self.bn1 = nn.BatchNorm2d(channels)
        self.bn3 = nn.BatchNorm2d(channels)
        self.bn5 = nn.BatchNorm2d(channels)
        self.act = nn.SiLU()
        
        # Spatial attention generator for each scale
        # Uses depthwise conv to keep it light
        self.att_conv = nn.Conv2d(channels * 3, 3, kernel_size=3, padding=1, groups=3)
        self.att_conv.weight.data.normal_(0, 0.01)
        self.att_conv.bias.data.zero_()
        
        # Final refinement
        self.fuse_conv = nn.Conv2d(channels, channels, 3, 1, 1)
        self.fuse_bn = nn.BatchNorm2d(channels)
        
    def forward(self, x):
        # Generate multi‑scale features
        x1 = self.act(self.bn1(self.conv1(x)))   # 1x1
        x3 = self.act(self.bn3(self.conv3(x)))   # 3x3
        x5 = self.act(self.bn5(self.conv5(x)))   # 5x5
        
        # Concatenate along channel to compute spatial weights
        cat = torch.cat([x1, x3, x5], dim=1)          # [B, 3*C, H, W]
        spatial_weights = self.att_conv(cat)          # [B, 3, H, W]
        spatial_weights = torch.softmax(spatial_weights, dim=1)  # sum to 1 per pixel
        
        # Fuse: weighted sum over the 3 scales
        out = (spatial_weights[:, 0:1, :, :] * x1 +
               spatial_weights[:, 1:2, :, :] * x3 +
               spatial_weights[:, 2:3, :, :] * x5)
        
        # Refine
        out = self.act(self.fuse_bn(self.fuse_conv(out)))
        
        # Residual connection
        return out + x


class C2f_SDF(C2f):
    """C2f with Spatial Dynamic Fusion (replaces original C2f)"""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.fusion = SpatialDynamicFusion(c2)   # <── NOVEL BLOCK

    def forward(self, x):
        # Standard C2f forward
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        out = self.cv2(torch.cat(y, 1))
        # Apply novel fusion
        return self.fusion(out)

# ------------------------------
# 4. Build Model and Inject Fusion Blocks
# ------------------------------
model = YOLO(model_name)
torch_model = model.model.model

def replace_neck_with_sdf(torch_model):
    # Find the SPPF layer (neck starts after it)
    sppf_idx = None
    for i, layer in enumerate(torch_model):
        if 'SPPF' in str(layer.__class__.__name__):
            sppf_idx = i
            break
    
    if sppf_idx is None:
        print("⚠️ SPPF not found – check model structure")
        return
    
    replaced = 0
    for i in range(sppf_idx + 1, len(torch_model)):
        layer = torch_model[i]
        if layer.__class__.__name__ in ['C2f', 'C3k2']:
            # Get output channels
            if hasattr(layer, 'cv2'):
                c_out = layer.cv2.conv.out_channels
            else:
                c_out = 256   # fallback
            
            # Replace with our C2f_SDF
            new_layer = C2f_SDF(c_out, c_out, n=len(layer.m) if hasattr(layer, 'm') else 1)
            
            # Copy weights where possible
            with torch.no_grad():
                try:
                    new_layer.cv1.load_state_dict(layer.cv1.state_dict())
                    new_layer.cv2.load_state_dict(layer.cv2.state_dict())
                    # Copy first few blocks
                    for j in range(min(len(new_layer.m), len(layer.m))):
                        new_layer.m[j].load_state_dict(layer.m[j].state_dict())
                except Exception as e:
                    print(f"  (weight copy warning: {e})")
            
            torch_model[i] = new_layer
            replaced += 1
            print(f"✓ Replaced layer {i} with C2f_SDF (spatial dynamic fusion)")
    
    print(f"✅ Injected {replaced} novel fusion blocks into YOLO11{model_size}")

replace_neck_with_sdf(torch_model)

# ------------------------------
# 5. Training on 100% of URPC2020 Dataset (30 epochs)
# ------------------------------
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=512,
    batch=batch,
    fraction=1.0,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project='urpc_yolo11_sdf',
    name=f'full_{model_size}_spatial_dynamic_fusion_100pct',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    augment=True,
    weight_decay=0.0005,
    patience=15
)

# ------------------------------
# 6. Evaluation (no inference on test images)
# ------------------------------
metrics = model.val(data=data_yaml, split='val', iou=0.5, conf=0.25)

print("\n" + "="*60)
print(f"FINAL EVALUATION - YOLO11{model_size} + Spatial Dynamic Fusion (100% Dataset, 30 epochs)")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")
f1 = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-6)
print(f"F1-Score     : {f1:.4f}")
print("="*60)

print(f"\n🎉 Done! YOLO11{model_size} + Novel Spatial‑wise Dynamic Fusion trained and evaluated on 100% of URPC2020 (30 epochs).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using model: yolo11m.pt
✅ Classes: ['echinus', 'holothurian', 'scallop', 'starfish'] (4 classes)
Dataset: /kaggle/working/URPC2020-2
  (weight copy warning: Error(s) in loading state_dict for Conv:
	size mismatch for conv.weight: copying a param with shape torch.Size([512, 1024, 1, 1]) from checkpoint, the shape in current model is torch.Size([512, 512, 1, 1]).)
✓ Replaced layer 13 with C2f_SDF (spatial dynamic fusion)
  (weight copy warning: Error(s) in loading state_dict for Conv:
	size mismatch for conv.weight: copying a param with shape torch.Size([256, 1024, 1, 1]) from checkpoint, the shape in curre